# Robot control - introduction to Kalman filter

Welcome to the brief journey into the world of filters.
In this notebook, we'll guide you through simple, step by step examples to build a solid understanding of the principles behind the Kalman filter.

The content is inspired by https://www.kalmanfilter.net/, and we will use the same notation as the authors.  While it's not required to read the blog in order to complete the exercise, doing so is recommended for curious readers who would like to explore the topic further.

Disclaimer: the examples given in this notebook are somewhat artificial in that we generate all measurements upfront and even store our predictions (for plotting purposes). This goes against the principle of filters, which great advantage is that we can obtain the same results using only data from current and previous iterations. However, all computations in this notebook will be perfomed using the Kalman equations.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

# Function for plotting
def plot_measurement_and_estimates(measurements, estimates, ground_truth):
    plt.figure(figsize=(10, 5))
    if measurements is not None:
        plt.plot(measurements, label="Noisy Measurements", marker='', linestyle='--')
    plt.plot(estimates, label="Filtered Estimates", marker='', color='r')
    plt.plot(ground_truth, label="Ground Truth", marker='', color='g')
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title("Measurements vs. Filtered Estimates")
    plt.legend()
    plt.grid(True)
    plt.show()

# Mean estimator as the simplest filter

Let's consider a simple example of measuring a building's height. We could take multiple measurements, and the best estimate of its height at any given point would be the average of all the received values.

<img src = 'https://drive.google.com/uc?id=1Slp72sYE8jellEWUz-YvtXa02Ni5BOcX' width = 400>

Before we proceed, let's define the following terms: <br>
$x$ is the true value of the height <br>
$z_{n}$ is the measured value of the height at time $n$ <br>
$\hat{x}_{n,n}$  is the estimate of $x$ at time $n$ (the estimate is made after taking the measurement $z_n$) <br>
$\hat{x}_{n,n-1}$  is the estimate of the future state of $x$ at time $n$ made at the time $n-1$

The equation for the best estimate can be transformed in the following way:
$\hat{x}_{n,n} = \frac{1}{n} \sum_{i=1}^{n} z_i = \frac{1}{n} \sum_{i=1}^{n-1} (z_i) + \frac{1}{n} z_n = \frac{1}{n} \frac{n-1}{n-1} \sum_{i=1}^{n-1} (z_i) + \frac{1}{n}z_n = \frac{n-1}{n} \hat{x}_{n-1,n-1} + \frac{1}{n}z_n = \hat{x}_{n-1,n-1} + \frac{1}{n} (z_n - \hat{x}_{n-1,n-1})$

The height of the building (hopefully) remains constant over the course of taking measurements, so $\hat{x}_{n,n-1} = \hat{x}_{n-1,n-1}$. This allows us to rewrite the equation as: <br>
$\hat{x}_{n,n} = \hat{x}_{n,n-1} + \frac{1}{n} (z_n - \hat{x}_{n,n-1})$

This equation is one of the Kalman filter equations known as the State Update Equation.

<img src = 'https://drive.google.com/uc?id=13O_LJ79BGvTCY3K3u-3ohqhKm5B6LaE6' width = 900>

- State Update Equation can be interpreted as a balance between our prior belief about the value and the newly observed value. Note that the term $\frac{n-1}{n} \hat{x}_{n,n-1} + \frac{1}{n}z_n$ represents a weighted average of these values. This concept of combining prior and observed information using a weighted average is fundamental and will reappear in more sophisticated filters.
- The advantage of this approach is that it eliminates the need to store all the measurements, requiring only the most recent set of measurements from the latest iteration.
- Factor $\frac{1}{n}$ is specific to this example. In general, the factor is called Kalman gain, and is usually denoted as $K_n$ or $\alpha_n$. <br>
- It's worth noting that the term $(z_n - \hat{x}_{n-1,n-1})$ is a difference between expectation and observed value, and is refered to as measurement residual or innovation. <br>


## Estimating building's height

**Task 0:** Let's begin with an initial guess of the state estimate $\hat{x}_{0,0} = 33.$ Using the State Update Equation complete the calculation of a current estimate of the state $x$. Discuss whether the rate of convergence to the ground truth depends on the choice of `initial_guess`.

In [ ]:
# Generate noisy measurements
num_measurements = 50
ground_truth = 30  # true height of the building
measurement_variance = 1  # variance of measurements
measurements = np.random.normal(ground_truth, np.sqrt(measurement_variance), num_measurements)

In [ ]:
initial_guess = 33
estimates = [initial_guess]

for n in range(1,num_measurements + 1):
    z = measurements[n - 1]  # current observation. measurement[n-1] corresponds to measurement taken at time n
    x_prev = estimates[-1]  # prediction of current state

    ### YOUR CODE STARTS ###
    # Calculate a current etimate of x using the State Update Equation
    x_now = ...

    ### YOUR CODE ENDS ###

    estimates.append(x_now)

plot_measurement_and_estimates(measurements, estimates[1:],np.full(num_measurements, ground_truth))

# The alpha-beta filter

Let's consider a dynamic system now, such as an airplane moving at a constant speed along one dimension e.g. moving away from a radar. Let $x$ be a distance from the radar.

<img src = 'https://drive.google.com/uc?id=1SDj8voXv2-FuzuVDycUm-7Af5uL9y4ns' width = 500>


In this case, we'll need two State Update Equations (one for position and the other for velocity): <br>
$\hat{x}_{n,n} = \hat{x}_{n,n-1} + \alpha (z_n - \hat{x}_{n,n-1})  $ <br>
$\hat{\dot{x}}_{n,n} = \hat{\dot{x}}_{n,n-1} + \beta (\frac{z_n - \hat{x}_{n,n-1}}{Δt})$  

$\alpha,\beta \in [0, 1]$

We can predict the next state with:

$\hat{x}_{n+1} = \hat{x}_{n} + Δt ⋅ \hat{\dot{x}}_n$ <br>
$\hat{\dot{x}}_{n+1} = \hat{\dot{x}}_n$

**Task 1a:** Complete the code below. First, update the current belief about the system's state, then predict the state for the next iteration using the formulas provided above.

**Task 1b** Run the code. Experiment with different values of $\alpha$ and $\beta$, and observe the resulting plots. Discuss how changes in these parameters impact the estimates. Under what circumstances would high values of $\alpha$ and $\beta$ be preferable?

In [ ]:
num_measurements_air = 50
true_position = 1000  # true initial position of the airplane
true_velocity = 50  # true constant velocity (m/s)
measurement_noise_variance = 90000  # variance of the measurement noise
delta_t = 3  # time interval at which measurements were taken

# Generate noisy measurements based on true position and velocity
times = np.arange(num_measurements_air)
ground_truth_air = true_position + true_velocity * times * delta_t
measurements_air = ground_truth_air + np.random.normal(0, np.sqrt(measurement_noise_variance), num_measurements_air)

In [ ]:
alpha = 0.1  # play with this number
beta = 0.1  # play with this number

 # Initializes estimates
estimated_positions = [1000]  # initial position estimate
estimated_velocities = [100]  # initial velocity estimate

x_prediction = estimated_positions[0]
x_dot_prediction = estimated_velocities[0]

for n in range(1, num_measurements_air + 1):
    z = measurements_air[n - 1]  # current observation (measurement[n-1] corresponds to measurement taken at time n)
    x_prior_belief = x_prediction  # prediction from previous iteration becomes our prior belief
    x_dot_prior_belief = x_dot_prediction  # prediction from previous iteration becomes our prior belief

    ### YOUR CODE STARTS ###
    x_current_belief = ...
    x_dot_current_belief = ...

    x_prediction = ...
    x_dot_prediction = ...
    ### YOUR CODE ENDS ###

    estimated_positions.append(x_current_belief)
    estimated_velocities.append(x_dot_current_belief)

plot_measurement_and_estimates(measurements_air,estimated_positions,ground_truth_air)


# One dimensional Kalman filter

Let's revisit our building example from Task 0. This time we carefully inspected our measuring device and discovered that the variance of each measurement is equal to 1. We'd like to incorporate this information into our model.

We can treat $\hat{x}_{n,n}$ and $\hat{x}_{n,n-1}$ as random variables. Let's denote: <br>
$r_n$ is the variance of measurement $z_n$ <br>
$p_{n,n}$ is the variance of the optimum estimate $\hat{x}_{n,n}$ <br>
$p_{n,n-1}$ is the variance of the prior estimate $\hat{x}_{n,n-1}$

The current estimate is a weighted average of observation and prior estimate.

$\hat{x}_{n,n} = w_1 z_n + (1 - w_1) \hat{x}_{n,n-1}$

Notice that by rearanging terms, we get the State Update Equation.

$\hat{x}_{n,n} = \hat{x}_{n,n-1} + w_1 (z_n - \hat{x}_{n,n-1})$

**Task 2a:** We can apply variance to both sides of equation mentioned above. Using your knowledge of variance properties, transform the equation further. Treat $w_1$ as a constant.

$\text{Var}(x_{n,n}) = \text{Var}(w_1 z_n + (1 - w_1) \hat{x}_{n,n-1})$

**Task 2b:** The optimal filter should minimize the variance of estimates. Find $w_1$ that minimizes $p_{n,n}$.

<img src = 'https://drive.google.com/uc?id=12oV2IKMXShztgpFpNHFurzy_-fGZRT6P' width = 500>

We are going to call the optimal weight the Kalman gain (denoted $K_n$).

$\hat{x}_{n,n} = \hat{x}_{n,n-1} + \frac{p_{n,n-1}}{p_{n,n-1} + r_n} (z_n - \hat{x}_{n,n-1})$

The final equation from Task 2a takes form:

$p_{n,n} = K_n^2r_n + (1-K_n)^2 p_{n,n-1}$

It can be transformed to the simpler form:

$p_{n,n} = (1-K_n)p_{n,n-1}$

This equation updates the estimate variance of the current state and is called the Covariance Update Equation.


**Task 2c:** Complete the code below to calculate estimates of the current state, as well as update the estimated variance of the current state.
We assume an initial guess of 33 meters and a human estimation error (standard deviation) of 5 meters. The measurement variance is `r = 1`.

In [ ]:
initial_guess = 33
variance_now = 5**2
r = 1
estimates = [initial_guess]

for n in range(1,num_measurements + 1):
    z = measurements[n - 1]  # current observation (measurement[n-1] corresponds to measurement taken at time n)
    x_prev = estimates[-1]  # prediction of current state
    variance_previous = variance_now

    ### YOUR CODE STARTS ###
    K_n = ...
    x_now = ...
    variance_now = ...

    ### YOUR CODE ENDS ###

    estimates.append(x_now)

print(f"The last state estimate: {x_now:.5f}")
print(f"The variance of the last estimate: {variance_now:.5f}")
plot_measurement_and_estimates(measurements, estimates[1:],np.full(num_measurements, ground_truth))

# Process noise

In the real world, processes often involve uncertainties, meaning the model we choose might not perfectly match what actually happens. In our tower-measurement example, suppose construction is still ongoing and the tower grows by about 1 meter between consecutive measurements. If we fail to include this in our model and instead assume the height stays constant, this mismatch introduces what we call process noise.

<img src = 'https://drive.google.com/uc?id=1anfKHt78hoLy8PIRGe3eZz2IyA8siAO3' width = 400>

One might argue that we could fix this by adding a “construction velocity” term to the model. But in reality, there would almost always be some randomness—for example, the construction velocity would probably vary in an unpredictable way.

Let’s denote the process noise variance by $q$. Then before any state update, we simply add this process noise variance to the predicted variance: <br>
$p_{n,n-1} = p_{n-1,n-1} + q$ <br>
The other equations from previous example remain unchanged.


**Task 3a:** Below is a more standard and commonly used implementation of the Kalman filter. Complete the `step` method.

**Task 3b:** Experiment with different values of `process_variance`. What happens when the value is too small or too large?

In [ ]:
# Generate growing tower measurements
measurements_growing = measurements + np.arange(num_measurements)
ground_growing = ground_truth + np.arange(num_measurements)

In [ ]:
class KalmanFilter1D:
    def __init__(self, initial_state, initial_variance, measurement_variance, process_variance):
        """
        1D Kalman Filter

        initial_state       = initial guess x₀
        initial_variance    = initial uncertainty p₀
        measurement_variance= measurement noise r
        process_variance    = process noise q
        """
        self.x = initial_state
        self.p = initial_variance
        self.r = measurement_variance
        self.q = process_variance

    def step(self, z):
        """
        Perform one Kalman filter update with measurement z.
        """

        ### YOUR CODE STARTS ###
        # ----- Predict -----
        x_pred = ...
        p_pred = ...

        # ----- Update -----
        K = ...
        self.x = ...
        self.p = ...
        ### YOUR CODE ENDS ###

        return self.x, self.p

In [ ]:
kf = KalmanFilter1D(
    initial_state=33,
    initial_variance=5**2,
    measurement_variance=1,
    process_variance=0) # experiment with this value

estimates = []

for z in measurements_growing:
    x, variance = kf.step(z)
    estimates.append(x)

print(f"The last state estimate: {x:.5f}")
print(f"The variance of the last estimate: {variance:.5f}")

plot_measurement_and_estimates(measurements_growing, estimates, np.full(num_measurements, ground_growing))